In [23]:
import pandas as pd
import numpy as np

In [24]:
# Import the dataset
df = pd.read_excel("ABS Tech Case 2026_Data.xlsx")

# Capitalize entries in column for hispanic/latino
df["HispanicLatino"] = df["HispanicLatino"].str.capitalize()

In [27]:
# Adding the calculated scores to the DataFrame
df["technical_score"] = ( 0.15 * df["TechLev"] + 0.10 * df["TrainHours"] +0.05 * df["AIUse"] + 0.05 * df["AIConf"] +0.10 * df["InnoCont"] +0.10 * df["SpecialProjectsCount"])
df["personal_score"] = (0.05 * df["EngagementSurvey"] +0.05 * df["WLF"] +0.15 * df["PerfScore"] +0.05 * df["JobStr"])
df["interpersonal_score"] = (0.12 * (df["Feedback"] + df["Trust"]) / 2 +0.06 * df["TeamIden"] +0.05 * df["Network"] +0.02 * df["ProjColl"])
df["talent_score"] = (0.50 * df["technical_score"] +0.25 * df["personal_score"] +0.25 * df["interpersonal_score"])

# Define the talent threshold
threshold = df["talent_score"].quantile(0.80)
df["is_talent"] = (df["talent_score"] >= threshold).astype(int)

## Question 1

In [35]:
def talent_demographic_summary(data):

    talent_col = "is_talent"

    demographics = [
        "GenderID",
        "RaceDesc",
        "HispanicLatino",
        "Department",
        "ManPos",
        "Remote",
        "Position",
        "TechLev"
    ]
    
    results = {}
    total_employees = len(df)
    total_talents = df[talent_col].sum()
    
    for var in demographics:
            
        summary = (
            df.groupby(var)
            .agg(
                total_employees=(talent_col, "count"),
                talents=(talent_col, "sum")
            )
        )
        
        summary["pct_talents_within_group"] = summary["talents"] / summary["total_employees"]
        summary["workforce_share"] = summary["total_employees"] / total_employees
        summary["talent_share_total"] = summary["talents"] / total_talents
        
        # Representation ratio:
        # % of all talents from group / % workforce from group
        summary["representation_ratio"] = (
            summary["talent_share_total"] / summary["workforce_share"]
        )
        
        summary = summary.sort_values("pct_talents_within_group", ascending=False)
        
        results[var] = summary.reset_index()
    
    return results

demographic_summary = talent_demographic_summary(df)

# demographic_summary["GenderID"]
# demographic_summary["RaceDesc"]
# demographic_summary["HispanicLatino"]
# demographic_summary["Department"]
# demographic_summary["ManPos"]
# demographic_summary["Remote"]
# demographic_summary["Position"]
# demographic_summary["TechLev"]


## Question 2

In [ ]:
def turnover1(data, group_col):
    tmp = data.copy()
    summary = (
        tmp.groupby(group_col)["Termd"]
           .agg(count= "size",leavers= "sum",stayers= lambda x: (1 - x).sum(),turnoverrate = "mean")
    )
    summary["turnoverrate"] = (summary["turnoverrate"]* 100)
    summary = summary.sort_values("turnoverrate",ascending=False)
    return summary

In [36]:
department = turnover1(df, "Department")
department

,count,leavers,stayers,turnoverrate
Department,,,,
Production,209,83,126,39.712919
Software Engineering,11,4,7,36.363636
Admin Offices,9,2,7,22.222222
IT/IS,50,10,40,20.000000
Sales,31,5,26,16.129032
Executive Office,1,0,1,0.000000
